# D8: unlearning with proxy English the model never saw

**Upload first:** the latest `colab/code_bundle.tar.gz` to `MyDrive/unlearn/`.
This runs the selected M3 configuration again (lr 5e-5, λ 2), this time using English that no model has trained on:
`en_relearn_fresh` from character 1M on. Then it evaluates the practitioner and oracle points on the test splits
(each read once) and runs both relearning attacks from them. About 30 min on a T4.
Every step stops with an error if its script fails.

In [ ]:
# 1. Settings + GPU check
SEED = 0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive. MyDrive/unlearn/{checkpoints,results} is the working copy from Phase 3 on.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os
for d in ('checkpoints', 'results', 'gate_packs'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Data bundle (unchanged since Phase 0) + latest code bundle on top; check the data SHAs;
#    link checkpoints/ and results/ to Drive so every script writes there directly.
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf {DRIVE}/unlearn_bundle.tar.gz -C /content/unlearn
!tar xzf {DRIVE}/code_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c --quiet MANIFEST.sha256 && echo "data OK"
!rm -rf checkpoints results && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/results results

In [ ]:
# 4. Dependencies + tests. tests/test_data.py is not run here: it needs the *_titles.txt files, which are
#    not in the bundle. It passed on the Mac at Gate 0, and cell 3 already proved the .bin files are
#    byte-identical to the Mac copy (SHA-256). Smoke-data tests are skipped (no smoke data on Colab).
!pip -q install pyyaml
!python -m pytest -q --ignore=tests/test_data.py 2>&1 | tail -2

In [ ]:
# 5. Bring the Mac's results into Drive (never overwrites), then re-hash every protected checkpoint.
#    Drive cannot chmod, so this SHA check is what protects θ_ref. It must print OK, never raise.
!python scripts/colab_sync.py merge --src results_mac --dst results
!python scripts/colab_sync.py verify

In [ ]:
# 6. Helper: run a script, stream its output, and raise if it fails.
import subprocess, sys, json
def run(*args):
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line:
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed with exit code {p.returncode}: {args}')
PROXY_POINTS = f'results/operating_points_proxy_s{SEED}.json'

In [ ]:
# 7. Proxy unlearning run (collapses within ~70 steps; dense eval every 5 steps)
run('scripts/unlearn.py', '--grid', '--config', 'configs/unlearn_proxy.yaml', '--seed', str(SEED))

In [ ]:
# 8. Operating points of the proxy run (own file: the Gate 4 selection is untouched), compared with the original M3
run('scripts/select_points.py', '--seed', str(SEED), '--config', 'configs/unlearn_proxy.yaml', '--out', PROXY_POINTS)
orig = json.load(open(f'results/operating_points_s{SEED}.json'))
prox = json.load(open(PROXY_POINTS))
both = {**prox, 'runs': {orig['selected']['M3']: orig['runs'][orig['selected']['M3']], **prox['runs']},
        'selected': {'M3': orig['selected']['M3'], 'M3p': prox['selected']['M3']}}
json.dump(both, open(f'results/operating_points_m3_vs_proxy_s{SEED}.json', 'w'), indent=1)
run('scripts/plots.py', 'trajectory', '--seed', str(SEED), '--points', f'results/operating_points_m3_vs_proxy_s{SEED}.json',
    '--out', f'results/figures/trajectory_proxy_s{SEED}.png')
from IPython.display import Image, display; display(Image(f'results/figures/trajectory_proxy_s{SEED}.png'))

In [ ]:
# 9. Test-split evaluation of the proxy run's practitioner and oracle points (earlier models are skipped)
run('scripts/phase5_eval.py', '--seed', str(SEED), '--points', PROXY_POINTS)

In [ ]:
# 10. Relearning attacks from the proxy run (theta_ref's runs are reused)
run('scripts/relearn.py', '--seed', str(SEED), '--points', PROXY_POINTS, '--suffix', 'proxy', '--with-oracle')

In [ ]:
# 11. Comparison table + relearning figure. Paste the table into the chat.
m3, m3p = orig['selected']['M3'], prox['selected']['M3']
names = [f'theta_ref_s{SEED}@test', f'theta_b_s{SEED}@test', f'{m3}@prac', f'{m3}@oracle', f'{m3p}@prac', f'{m3p}@oracle']
run('scripts/plots.py', 'table', '--names', *names, '--out', 'results/table_proxy.md')
run('scripts/plots.py', 'relearn', '--seed', str(SEED), '--labels', 'theta_ref', 'M3', 'M3_oracle', 'M3_proxy', 'M3_proxy_oracle',
    '--out', f'results/figures/relearn_proxy_s{SEED}.png')
display(Image(f'results/figures/relearn_proxy_s{SEED}.png'))

In [ ]:
# Gate pack: re-check the hashes, then zip results/ (small files only, no checkpoints).
# Download MyDrive/unlearn/gate_packs/gate6_results.zip and unzip it into the project as colab_results/gate6/
!python scripts/colab_sync.py verify
!cd {DRIVE} && rm -f gate_packs/gate6_results.zip && zip -qr gate_packs/gate6_results.zip results && ls -la gate_packs